# 🤖 Machine Learning Demo Apps
## Time Series Forecasting · Customer Segmentation · CNN Image Classification

Notebook ini berisi tiga aplikasi Streamlit interaktif untuk kelas Machine Learning.

📖 **Dokumentasi lengkap:** [github.com/afrizalmeka/ml-demo-apps](https://github.com/afrizalmeka/ml-demo-apps)

| App | Deskripsi | Port |
|-----|-----------|------|
| 🔮 **Time Series Forecasting** | Peramalan deret waktu dengan ML klasik + Deep Learning | 8501 |
| 👥 **Customer Segmentation** | Segmentasi pelanggan dengan KMeans Clustering | 8502 |
| 🧠 **CNN Image Classification** | Klasifikasi gambar dengan CNN PyTorch — Sesi 4 | 8503 |

### Langkah Penggunaan:
1. Jalankan **Cell 1** — install dependencies
2. Jalankan **Cell 2** — masukkan ngrok authtoken Anda
3. Jalankan **Cell 3** — tulis file `app_timeseries.py`
4. Jalankan **Cell 4** — tulis file `app_segmentasi.py`
5. Jalankan **Cell 5** — tulis file `app_cnn.py`
6. Jalankan **Cell 6** — jalankan **ketiga app sekaligus** → klik URL yang muncul
7. *(Opsional)* Jalankan **Cell 7** — untuk menghentikan semua tunnel ngrok

---
## ⚙️ CELL 1 — Install Dependencies

In [ ]:
!pip install streamlit pyngrok plotly scikit-learn pandas numpy yfinance torch torchvision Pillow

---
## 🔑 CELL 2 — Setup Ngrok Authtoken

> **Dapatkan token gratis di:** https://dashboard.ngrok.com/get-started/your-authtoken  
> Ganti `YOUR_NGROK_AUTHTOKEN_HERE` dengan token Anda.

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN_HERE")

---
## 🔮 CELL 3 — Tulis File `app_timeseries.py`

In [ ]:
%%writefile app_timeseries.py
# ============================================================
# APP 1: TIME SERIES FORECASTING DEMO
# Supervised Learning + Deep Learning untuk peramalan deret waktu
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ── Konfigurasi halaman ──────────────────────────────────────
st.set_page_config(
    page_title="🔮 Time Series Forecasting Demo",
    page_icon="🔮",
    layout="wide"
)

st.title("🔮 Time Series Forecasting")
st.markdown(
    "Demo interaktif peramalan deret waktu — bandingkan **ML Klasik** "
    "(Linear Regression, Ridge, Decision Tree) dengan **Deep Learning** "
    "(LSTM, GRU, 1D CNN, LSTM+Attention). Cocok untuk memahami kapan deep learning benar-benar unggul."
)

# ── Konstanta ────────────────────────────────────────────────
KELOMPOK_MODEL = {
    "📐 ML Klasik": ["Linear Regression", "Ridge Regression", "Decision Tree"],
    "🧠 Deep Learning": ["LSTM", "GRU", "1D CNN (TCN-lite)", "LSTM + Attention"],
}

INFO_MODEL = {
    "Linear Regression": "Baseline sederhana. Cocok untuk tren linear, buruk untuk pola non-linear / volatile.",
    "Ridge Regression":  "Linear + regularisasi L2. Lebih stabil dari Linear Regression pada data noisy.",
    "Decision Tree":     "Menangkap non-linearitas, tapi rentan overfit pada data kecil.",
    "LSTM":              "Long Short-Term Memory — RNN yang dirancang untuk mengingat dependensi jangka panjang. Pilihan utama untuk data saham.",
    "GRU":               "Gated Recurrent Unit — versi LSTM yang lebih ringan dan sering setara akurasinya. Lebih cepat dilatih.",
    "1D CNN (TCN-lite)": "Konvolusi 1D temporal — menangkap pola lokal dalam window waktu. Sangat cepat, cocok untuk pola musiman.",
    "LSTM + Attention":  "LSTM dengan mekanisme Attention — model 'fokus' pada timestep yang paling relevan. Unggul pada data event-driven seperti saham.",
}

WARNA_MODEL = {
    "Linear Regression":  "#2196F3",
    "Ridge Regression":   "#4CAF50",
    "Decision Tree":      "#FF9800",
    "LSTM":               "#9C27B0",
    "GRU":                "#E91E63",
    "1D CNN (TCN-lite)":  "#F44336",
    "LSTM + Attention":   "#FF5722",
}

# ── Fungsi: data demo ────────────────────────────────────────
def buat_data_demo():
    rng = np.random.RandomState(42)
    n = 60
    tanggal = pd.date_range(start="2020-01", periods=n, freq="MS")
    tren = np.linspace(100, 300, n)
    seasonal = np.array([-20 if m in [1,2] else (40 if m in [11,12] else 0) for m in tanggal.month])
    nilai = tren + seasonal + rng.normal(0, 12, n)
    nilai[17] += 60
    nilai[35] -= 40
    return pd.DataFrame({"tanggal": tanggal, "nilai": np.round(nilai, 2)})

# ── Fungsi: lag features (ML Klasik) ─────────────────────────
def buat_lag_features(df, kolom, n_lag):
    d = df.copy()
    for i in range(1, n_lag + 1):
        d[f"lag_{i}"] = d[kolom].shift(i)
    return d.dropna().reset_index(drop=True)

# ── Fungsi: sequence (Deep Learning) ─────────────────────────
def buat_sequence(arr, seq_len):
    X, y = [], []
    for i in range(seq_len, len(arr)):
        X.append(arr[i-seq_len:i])
        y.append(arr[i])
    return np.array(X), np.array(y)

# ── Definisi model Deep Learning ────────────────────────────
class ModelLSTM(nn.Module):
    def __init__(self, hidden=64, n_layer=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, n_layer, batch_first=True,
                            dropout=dropout if n_layer > 1 else 0.0)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

class ModelGRU(nn.Module):
    def __init__(self, hidden=64, n_layer=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(1, hidden, n_layer, batch_first=True,
                          dropout=dropout if n_layer > 1 else 0.0)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

class ModelTCN(nn.Module):
    def __init__(self, n_filter=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, n_filter, kernel_size=3, padding=2, dilation=1), nn.ReLU(),
            nn.Conv1d(n_filter, n_filter, kernel_size=3, padding=4, dilation=2), nn.ReLU(),
            nn.Conv1d(n_filter, n_filter, kernel_size=3, padding=8, dilation=4), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Linear(n_filter, 1)
    def forward(self, x):
        return self.fc(self.net(x.permute(0,2,1)).squeeze(-1)).squeeze(-1)

class AttensiLayer(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.W = nn.Linear(hidden, hidden)
        self.v = nn.Linear(hidden, 1, bias=False)
    def forward(self, h):
        alpha = torch.softmax(self.v(torch.tanh(self.W(h))), dim=1)
        return (alpha * h).sum(dim=1)

class ModelLSTMAttention(nn.Module):
    def __init__(self, hidden=64, n_layer=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, n_layer, batch_first=True,
                            dropout=dropout if n_layer > 1 else 0.0)
        self.attn = AttensiLayer(hidden)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.attn(out)).squeeze(-1)

# ── Fungsi: training DL ───────────────────────────────────────
def train_dl(model, X_tr, y_tr, X_val, y_val, lr, n_epoch, batch_size, device, prog=None):
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    crit = nn.MSELoss()
    loader = DataLoader(
        TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                      torch.tensor(y_tr, dtype=torch.float32)),
        batch_size=batch_size, shuffle=True
    )
    Xv = torch.tensor(X_val, dtype=torch.float32).to(device)
    yv = torch.tensor(y_val, dtype=torch.float32).to(device)
    best_val, best_state = float('inf'), None
    for ep in range(1, n_epoch + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = crit(model(Xv), yv).item()
        sched.step(vl)
        if vl < best_val:
            best_val = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if prog:
            prog.progress(ep/n_epoch, text=f"Epoch {ep}/{n_epoch} — val loss: {vl:.6f}")
    model.load_state_dict(best_state)
    return model

def prediksi_dl_iteratif(model, scaler, nilai_hist, seq_len, n_steps, device):
    model.eval()
    buf = list(scaler.transform(nilai_hist.reshape(-1,1)).flatten())
    hasil = []
    with torch.no_grad():
        for _ in range(n_steps):
            x = torch.tensor(buf[-seq_len:], dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
            p = model(x).item()
            hasil.append(p)
            buf.append(p)
    return scaler.inverse_transform(np.array(hasil).reshape(-1,1)).flatten()

def hitung_metrik(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    return mae, rmse, r2

def hitung_naive(y_true, last_train):
    y_naive = np.concatenate([[last_train], y_true[:-1]])
    return hitung_metrik(y_true, y_naive)

def buat_tanggal_fc(tgl_terakhir, n, interval):
    hasil, tgl = [], pd.Timestamp(tgl_terakhir)
    for _ in range(n):
        if "harian" in interval:   tgl += pd.tseries.offsets.BDay(1)
        elif "mingguan" in interval: tgl += pd.DateOffset(weeks=1)
        else:                        tgl += pd.DateOffset(months=1)
        hasil.append(tgl)
    return hasil


# ═══════════════════════════════════════════════════════════════
# SIDEBAR
# ═══════════════════════════════════════════════════════════════
with st.sidebar:
    st.header("⚙️ Pengaturan")

    sumber_data = st.radio("Pilih Sumber Data", [
        "📊 Data Demo (Penjualan Sintetis)",
        "📈 Data Saham Live (via yfinance)",
        "📁 Upload CSV Sendiri"
    ])

    df_data, nama_dataset, interval_data = None, "Data Demo", "bulanan"

    if sumber_data == "📊 Data Demo (Penjualan Sintetis)":
        df_data = buat_data_demo()
        nama_dataset = "Penjualan Sintetis"

    elif sumber_data == "📈 Data Saham Live (via yfinance)":
        ticker_input = st.text_input("Kode Saham", value="BBCA.JK",
                                     placeholder="BBCA.JK / TLKM.JK / AAPL / TSLA")
        periode_map  = {"1 Tahun":"1y","2 Tahun":"2y","3 Tahun":"3y","5 Tahun":"5y"}
        periode_label = st.selectbox("Periode Data", list(periode_map.keys()), index=1)
        interval_map  = {"Harian":"1d","Mingguan":"1wk","Bulanan":"1mo"}
        interval_label = st.selectbox("Interval", list(interval_map.keys()))

        if st.button("🔍 Ambil Data"):
            try:
                raw = yf.download(ticker_input.strip(), period=periode_map[periode_label],
                                  interval=interval_map[interval_label], progress=False)
                if raw.empty or "Close" not in raw.columns:
                    raise ValueError(f"Ticker '{ticker_input}' tidak ditemukan.")
                raw = raw.reset_index()
                col_tgl = "Datetime" if "Datetime" in raw.columns else "Date"
                df_s = raw[[col_tgl, "Close"]].copy()
                df_s.columns = ["tanggal", "nilai"]
                df_s["nilai"] = pd.to_numeric(df_s["nilai"], errors="coerce")
                df_s = df_s.dropna().reset_index(drop=True)
                st.session_state.update({"df_saham": df_s, "nama_saham": ticker_input.strip(),
                                          "interval_saham": interval_label.lower()})
                st.info(f"✅ {len(df_s)} baris data **{ticker_input.strip()}** berhasil diambil")
            except Exception as e:
                st.error(f"❌ {e}")
                st.session_state.update({"df_saham": buat_data_demo(),
                                          "nama_saham": "Demo (fallback)", "interval_saham": "bulanan"})

        if "df_saham" in st.session_state:
            df_data = st.session_state["df_saham"]
            nama_dataset = f"Saham {st.session_state.get('nama_saham','')}"
            interval_data = st.session_state.get("interval_saham","harian")
        else:
            df_data, nama_dataset = buat_data_demo(), "Data Demo (belum fetch)"

        st.info("💡 Data saham sangat noisy — **LSTM / LSTM+Attention** biasanya jauh lebih baik dari model linear.")

    else:
        st.markdown("**Format CSV:** kolom tanggal (`YYYY-MM` / `YYYY-MM-DD`) + kolom nilai numerik.")
        file_upload = st.file_uploader("Upload file CSV", type=["csv"])
        if file_upload:
            try:
                df_upload = pd.read_csv(file_upload)
                st.dataframe(df_upload.head(3))
                cols = list(df_upload.columns)
                col_t = st.selectbox("Kolom tanggal", cols, 0)
                col_v = st.selectbox("Kolom nilai",   cols, min(1, len(cols)-1))
                df_c = df_upload[[col_t, col_v]].rename(columns={col_t:"tanggal", col_v:"nilai"})
                df_c["tanggal"] = pd.to_datetime(df_c["tanggal"])
                df_c["nilai"]   = pd.to_numeric(df_c["nilai"], errors="coerce")
                df_c = df_c.dropna().sort_values("tanggal").reset_index(drop=True)
                if len(df_c) < 20:
                    raise ValueError("Minimal 20 baris data.")
                df_data, nama_dataset = df_c, file_upload.name
            except Exception as e:
                st.error(f"❌ {e}")
                df_data, nama_dataset = buat_data_demo(), "Data Demo (fallback)"
        else:
            df_data, nama_dataset = buat_data_demo(), "Data Demo (menunggu upload)"

    st.divider()

    # ── Pilih Model ──────────────────────────────────────────
    st.subheader("🤖 Pilih Model")
    model_terpilih = []
    for grup, daftar in KELOMPOK_MODEL.items():
        st.markdown(f"**{grup}**")
        for m in daftar:
            default = m in ["Linear Regression", "LSTM"]
            if st.checkbox(m, value=default, help=INFO_MODEL[m], key=f"cb_{m}"):
                model_terpilih.append(m)
    if not model_terpilih:
        st.warning("Pilih minimal 1 model!")
        model_terpilih = ["Linear Regression"]

    st.divider()
    st.subheader("📏 Parameter Umum")
    n_lag      = st.slider("Lag / Sequence Length", 5, 30, 10,
                            help="Jumlah timestep historis sebagai input. Berlaku untuk semua model.")
    n_forecast = st.slider("Jumlah Periode Forecast", 1, 30, 10)

    dl_dipilih = [m for m in model_terpilih if m in KELOMPOK_MODEL["🧠 Deep Learning"]]
    if dl_dipilih:
        st.divider()
        st.subheader("⚡ Hyperparameter Deep Learning")
        dl_epoch   = st.slider("Epoch", 20, 200, 50, step=10)
        dl_hidden  = st.selectbox("Hidden Size (LSTM/GRU)", [32, 64, 128], index=1)
        dl_lr      = st.select_slider("Learning Rate",
                                       [0.0001, 0.0005, 0.001, 0.005, 0.01], value=0.001)
        dl_batch   = st.selectbox("Batch Size", [16, 32, 64], index=1)
        dl_dropout = st.slider("Dropout", 0.0, 0.5, 0.2, step=0.1)
        with st.expander("ℹ️ Tips Deep Learning untuk Saham"):
            st.markdown("""
**LSTM** — pilihan utama untuk saham; bisa mengingat dependensi jangka panjang (tren multi-minggu).

**GRU** — lebih ringan dari LSTM, training lebih cepat, sering performa mirip. Cocok saat data terbatas.

**1D CNN** — menangkap pola berulang (pola mingguan/bulanan). Sangat cepat. Kurang baik pada tren panjang.

**LSTM + Attention** — terbaik untuk data event-driven (berita, earnings, krisis). Mekanisme *attention* membantu model fokus pada timestep kunci.

**Tips:**
- Saham harian → sequence length 20–30 (≈1 bulan trading)
- Saham mingguan → sequence length 10–15
- Tambah epoch jika val loss masih turun
- Turunkan LR jika loss naik-turun tidak stabil
            """)
    else:
        dl_epoch, dl_hidden, dl_lr, dl_batch, dl_dropout = 50, 64, 0.001, 32, 0.2

    tombol_run = st.button("🚀 Jalankan Semua Model", type="primary", use_container_width=True)


# ═══════════════════════════════════════════════════════════════
# SETUP DATA
# ═══════════════════════════════════════════════════════════════
if df_data is None or len(df_data) == 0:
    df_data, nama_dataset = buat_data_demo(), "Data Demo (fallback)"

if len(df_data) < n_lag + 5:
    st.error(f"Data terlalu sedikit ({len(df_data)} baris). Kurangi Sequence Length atau gunakan lebih banyak data.")
    st.stop()

st.info(f"📂 Sumber: **{nama_dataset}** — {len(df_data)} baris | Model aktif: **{', '.join(model_terpilih)}**")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
split_idx = max(int(len(df_data) * 0.8), n_lag + 2)

nilai_all   = df_data["nilai"].values.astype(float)
nilai_train = nilai_all[:split_idx]
nilai_test  = nilai_all[split_idx:]

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(nilai_train.reshape(-1, 1))

# Lag features untuk ML Klasik
df_lag     = buat_lag_features(df_data, "nilai", n_lag)
lag_cols   = [f"lag_{i}" for i in range(1, n_lag+1)]
split_lag  = int(len(df_lag) * 0.8)
df_lag_tr  = df_lag.iloc[:split_lag]
df_lag_te  = df_lag.iloc[split_lag:]
X_tr_ml, y_tr_ml = df_lag_tr[lag_cols].values, df_lag_tr["nilai"].values
X_te_ml, y_te_ml = df_lag_te[lag_cols].values, df_lag_te["nilai"].values
tgl_test_ml = df_lag_te["tanggal"].values

# Sequence untuk DL
val_sc     = scaler.transform(nilai_all.reshape(-1,1)).flatten()
X_seq, y_seq = buat_sequence(val_sc, n_lag)
split_seq  = max(split_idx - n_lag, 2)
X_tr_dl    = X_seq[:split_seq].reshape(-1, n_lag, 1)
y_tr_dl    = y_seq[:split_seq]
X_va_dl    = X_seq[split_seq:].reshape(-1, n_lag, 1)
y_va_dl    = y_seq[split_seq:]
tgl_te_dl  = df_data["tanggal"].values[n_lag + split_seq:]

tanggal_fc = buat_tanggal_fc(df_data["tanggal"].values[-1], n_forecast, interval_data)


# ═══════════════════════════════════════════════════════════════
# TRAINING
# ═══════════════════════════════════════════════════════════════
run_key = f"{nama_dataset}_{','.join(sorted(model_terpilih))}_{n_lag}_{n_forecast}_{dl_epoch}_{dl_hidden}_{dl_lr}_{dl_dropout}"

if tombol_run or "hasil_model" not in st.session_state or st.session_state.get("run_key") != run_key:
    hasil_model = {}

    for nama in model_terpilih:
        if nama in KELOMPOK_MODEL["📐 ML Klasik"]:
            if nama == "Linear Regression": m = LinearRegression()
            elif nama == "Ridge Regression": m = Ridge(alpha=1.0)
            else: m = DecisionTreeRegressor(max_depth=5, random_state=42)
            m.fit(X_tr_ml, y_tr_ml)
            pred_te = m.predict(X_te_ml)
            buf = list(nilai_all[-n_lag:])
            fc = []
            for _ in range(n_forecast):
                feat = np.array(buf[-n_lag:][::-1]).reshape(1,-1)
                p = m.predict(feat)[0]
                fc.append(p); buf.append(p)
            mae, rmse, r2 = hitung_metrik(y_te_ml, pred_te)
            hasil_model[nama] = dict(pred_test=pred_te, tgl_test=tgl_test_ml,
                                     y_test_aktual=y_te_ml, forecast=np.array(fc),
                                     mae=mae, rmse=rmse, r2=r2,
                                     warna=WARNA_MODEL[nama], tipe="ML Klasik")
        else:
            with st.status(f"🧠 Training **{nama}**...", expanded=False) as status_box:
                prog = st.empty()
                if nama == "LSTM":
                    mdl = ModelLSTM(dl_hidden, 2, dl_dropout)
                elif nama == "GRU":
                    mdl = ModelGRU(dl_hidden, 2, dl_dropout)
                elif nama == "1D CNN (TCN-lite)":
                    mdl = ModelTCN(n_filter=32)
                else:
                    mdl = ModelLSTMAttention(dl_hidden, 2, dl_dropout)

                mdl = train_dl(mdl, X_tr_dl, y_tr_dl, X_va_dl, y_va_dl,
                                dl_lr, dl_epoch, dl_batch, device, prog)
                status_box.update(label=f"✅ {nama} selesai", state="complete")

            mdl.eval()
            with torch.no_grad():
                pred_sc = mdl(torch.tensor(X_va_dl, dtype=torch.float32).to(device)).cpu().numpy()
            pred_te   = scaler.inverse_transform(pred_sc.reshape(-1,1)).flatten()
            y_te_act  = scaler.inverse_transform(y_va_dl.reshape(-1,1)).flatten()
            fc_vals   = prediksi_dl_iteratif(mdl, scaler, nilai_all, n_lag, n_forecast, device)
            mae, rmse, r2 = hitung_metrik(y_te_act, pred_te)
            hasil_model[nama] = dict(pred_test=pred_te, tgl_test=tgl_te_dl,
                                     y_test_aktual=y_te_act, forecast=fc_vals,
                                     mae=mae, rmse=rmse, r2=r2,
                                     warna=WARNA_MODEL[nama], tipe="Deep Learning")

    st.session_state["hasil_model"] = hasil_model
    st.session_state["run_key"] = run_key
    st.success(f"✅ {len(hasil_model)} model selesai dilatih!")

hasil_model = st.session_state.get("hasil_model", {})
if not hasil_model:
    st.info("⬅️ Pilih model dan klik **Jalankan Semua Model** untuk memulai.")
    st.stop()


# ═══════════════════════════════════════════════════════════════
# TABS
# ═══════════════════════════════════════════════════════════════
tab1, tab2, tab3, tab4 = st.tabs([
    "📈 Prediksi & Forecast",
    "📊 Perbandingan Model",
    "🔍 Data Mentah",
    "📚 Tentang Model DL",
])


# ── Tab 1: Prediksi & Forecast ────────────────────────────────
with tab1:
    model_tampil = st.multiselect("Tampilkan model di chart:", list(hasil_model.keys()),
                                   default=list(hasil_model.keys()))
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_data["tanggal"], y=df_data["nilai"], mode="lines", name="Aktual",
        line=dict(color="#607D8B", width=2),
        hovertemplate="%{x|%Y-%m-%d}<br>Nilai: %{y:.2f}<extra>Aktual</extra>"
    ))
    tgl_split_str = str(pd.Timestamp(df_data["tanggal"].values[split_idx]))
    fig.add_vline(x=tgl_split_str, line_dash="dot", line_color="gray")
    fig.add_annotation(x=tgl_split_str, y=1, yref="paper",
                       text="Train | Test", showarrow=False,
                       font=dict(color="gray", size=11),
                       xanchor="left", yanchor="top")
    for nama in model_tampil:
        h = hasil_model[nama]
        fig.add_trace(go.Scatter(
            x=h["tgl_test"], y=h["pred_test"], mode="lines",
            name=f"{nama} (test)", line=dict(color=h["warna"], width=1.5, dash="dash"),
            hovertemplate=f"%{{x|%Y-%m-%d}}<br>{nama}: %{{y:.2f}}<extra></extra>"
        ))
        fig.add_trace(go.Scatter(
            x=tanggal_fc, y=h["forecast"], mode="lines+markers",
            name=f"{nama} (forecast)", line=dict(color=h["warna"], width=2, dash="dot"),
            marker=dict(size=4),
            hovertemplate=f"%{{x|%Y-%m-%d}}<br>{nama} fc: %{{y:.2f}}<extra></extra>"
        ))
    fig.update_layout(title=f"Prediksi & Forecast — {nama_dataset}",
                      xaxis_title="Tanggal", yaxis_title="Nilai",
                      legend=dict(orientation="h", y=-0.25),
                      hovermode="x unified", height=520)
    st.plotly_chart(fig, use_container_width=True)

    st.subheader("📋 Tabel Forecast ke Depan")
    df_fc = pd.DataFrame({"Periode": [t.strftime("%Y-%m-%d") for t in tanggal_fc]})
    for nm in model_tampil:
        df_fc[nm] = np.round(hasil_model[nm]["forecast"], 2)
    st.dataframe(df_fc, use_container_width=True, hide_index=True)


# ── Tab 2: Perbandingan Model ─────────────────────────────────
with tab2:
    mae_b, rmse_b, r2_b = hitung_naive(df_lag_te["nilai"].values, df_lag_tr["nilai"].values[-1])

    rows = [{"Model":"Naive Baseline","Tipe":"Baseline",
             "MAE":round(mae_b,4),"RMSE":round(rmse_b,4),"R²":round(r2_b,4)}]
    for nm, h in hasil_model.items():
        rows.append({"Model":nm,"Tipe":h["tipe"],
                     "MAE":round(h["mae"],4),"RMSE":round(h["rmse"],4),"R²":round(h["r2"],4)})
    df_mtk = pd.DataFrame(rows).sort_values("RMSE")
    st.dataframe(df_mtk, use_container_width=True, hide_index=True)

    warna_bar = ["#B0BEC5"] + [WARNA_MODEL.get(n,"#999") for n in df_mtk["Model"].iloc[1:]]

    fig_mae = go.Figure(go.Bar(
        x=df_mtk["Model"], y=df_mtk["MAE"], marker_color=warna_bar,
        text=df_mtk["MAE"].map("{:.4f}".format), textposition="outside",
        hovertemplate="%{x}<br>MAE: %{y:.4f}<extra></extra>"
    ))
    fig_mae.update_layout(title="MAE (lebih kecil = lebih baik)",
                          xaxis_title="Model", yaxis_title="MAE", height=360)
    st.plotly_chart(fig_mae, use_container_width=True)

    fig_r2 = go.Figure(go.Bar(
        x=df_mtk["Model"], y=df_mtk["R²"], marker_color=warna_bar,
        text=df_mtk["R²"].map("{:.4f}".format), textposition="outside",
        hovertemplate="%{x}<br>R²: %{y:.4f}<extra></extra>"
    ))
    fig_r2.update_layout(title="R² (lebih besar = lebih baik)",
                         xaxis_title="Model", yaxis_title="R²",
                         yaxis=dict(range=[min(-0.5, df_mtk["R²"].min()-0.1), 1.1]), height=360)
    st.plotly_chart(fig_r2, use_container_width=True)

    best_nm = df_mtk.iloc[0]["Model"]
    best_rmse = df_mtk.iloc[0]["RMSE"]
    st.success(f"🏆 Model terbaik (RMSE terendah): **{best_nm}** — RMSE = `{best_rmse:.4f}`")

    if best_nm in KELOMPOK_MODEL["🧠 Deep Learning"]:
        st.info(
            f"✅ **{best_nm}** (Deep Learning) mengalahkan semua model ML klasik pada dataset ini. "
            "Ini menunjukkan pola data membutuhkan kemampuan memodelkan dependensi jangka panjang."
        )
    elif any(h["tipe"]=="Deep Learning" for h in hasil_model.values()):
        st.info("ℹ️ Model ML klasik masih kompetitif. Deep learning butuh lebih banyak data/epoch untuk menunjukkan keunggulannya.")


# ── Tab 3: Data Mentah ────────────────────────────────────────
with tab3:
    st.subheader(f"Data: {nama_dataset}")
    st.dataframe(df_data, use_container_width=True)
    st.download_button("⬇️ Download CSV",
                       df_data.to_csv(index=False).encode("utf-8"),
                       "data_timeseries.csv", "text/csv")


# ── Tab 4: Tentang Model DL ───────────────────────────────────
with tab4:
    st.subheader("📚 Panduan Memilih Model untuk Data Saham")

    tbl = [
        ["Linear Regression", "📐 ML Klasik", "Cepat, interpretable", "Tidak bisa non-linear", "Tren sederhana, edukasi"],
        ["Ridge Regression",  "📐 ML Klasik", "Stabil di data noisy", "Masih linear", "Data dengan multikolinearitas"],
        ["Decision Tree",     "📐 ML Klasik", "Non-linear, no scaling needed", "Mudah overfit", "Baseline non-linear"],
        ["LSTM",              "🧠 Deep Learning", "Ingat dependensi panjang, state-of-the-art saham", "Butuh banyak data & epoch", "Saham harian, data panjang"],
        ["GRU",               "🧠 Deep Learning", "Lebih cepat dari LSTM, sering setara akurasi", "Sedikit kurang kapasitas memori", "Saat data terbatas"],
        ["1D CNN (TCN-lite)", "🧠 Deep Learning", "Sangat cepat, tangkap pola lokal", "Kurang baik di dependensi panjang", "Pola musiman berulang"],
        ["LSTM + Attention",  "🧠 Deep Learning", "Fokus pada timestep kunci, terbaik event-driven", "Paling lambat dilatih", "Saham volatile, berita/earnings"],
    ]
    st.table(pd.DataFrame(tbl, columns=["Model","Kategori","Keunggulan","Kelemahan","Cocok untuk"]))

    st.subheader("🔬 Cara Kerja Tiap Model Deep Learning")
    c1, c2 = st.columns(2)
    with c1:
        st.markdown("""
**🔵 LSTM (Long Short-Term Memory)**
```
Input [x₁, x₂, ..., xₜ]
  ↓ forget gate → lupakan info lama
  ↓ input gate  → tambah info baru  
  ↓ output gate → output ke layer berikut
  ↓ Linear → prediksi harga tₜ₊₁
```
*Cell state* membawa info ratusan timestep ke belakang tanpa vanishing gradient.

---

**🟢 GRU (Gated Recurrent Unit)**
```
Mirip LSTM, hanya 2 gate:
  reset gate  → berapa banyak past "dilupakan"
  update gate → gabungkan past + present
```
Parameter lebih sedikit → training lebih cepat.
        """)
    with c2:
        st.markdown("""
**🔴 1D CNN (Temporal Convolutional Network-lite)**
```
Input (batch, seq_len, 1)
  ↓ Conv1d dilation=1  → pola 3 timestep
  ↓ Conv1d dilation=2  → pola 5 timestep
  ↓ Conv1d dilation=4  → pola 9 timestep
  ↓ GlobalAvgPool → Linear → prediksi
```
*Dilated convolution* memperluas receptive field secara eksponensial.

---

**🟠 LSTM + Attention**
```
LSTM → hidden states [h₁, h₂, ..., hₜ]
  ↓ score(hᵢ) = v·tanh(W·hᵢ)
  ↓ α = softmax(score)   ← bobot fokus
  ↓ context = Σ αᵢ·hᵢ   ← weighted sum
  ↓ Linear → prediksi
```
Model "belajar" timestep mana yang paling penting.
        """)

    st.subheader("📈 Rekomendasi Konfigurasi berdasarkan Jenis Data")
    st.markdown("""
| Jenis Data | Sequence Length | Model Rekomendasi | Epoch | Catatan |
|------------|-----------------|-------------------|-------|---------|
| Harian (1–2 tahun) | 20–30 | LSTM / LSTM+Attn | 80–150 | ≥200 data poin ideal |
| Mingguan (3–5 tahun) | 10–15 | GRU / LSTM | 50–100 | Lebih sedikit noise |
| Bulanan | 6–12 | 1D CNN / GRU | 30–60 | Data terbatas → model ringan |
| Intraday | 30–60 | TCN / LSTM+Attn | 100–200 | Normalisasi ketat diperlukan |
    """)


---
## 👥 CELL 4 — Tulis File `app_segmentasi.py`

In [ ]:
%%writefile app_segmentasi.py
# ============================================================
# APP 2: CUSTOMER SEGMENTATION DEMO
# Unsupervised Learning — KMeans Clustering
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ── Konfigurasi halaman ──────────────────────────────────────
st.set_page_config(
    page_title="👥 Customer Segmentation Demo",
    page_icon="👥",
    layout="wide"
)

st.title("👥 Customer Segmentation")
st.markdown(
    "Demo interaktif segmentasi pelanggan menggunakan "
    "Unsupervised Learning (KMeans Clustering). "
    "Geser slider K dan eksplorasi bagaimana cluster terbentuk."
)

# ── Fungsi: buat data demo ───────────────────────────────────
def buat_data_demo():
    """Generate 200 pelanggan sintetis dengan 3 cluster, seed=42."""
    rng = np.random.RandomState(42)

    # Cluster A: Pelanggan Pasif
    n_a = 60
    usia_a = rng.uniform(20, 30, n_a)
    pengeluaran_a = rng.uniform(200, 400, n_a)
    frekuensi_a = rng.uniform(1, 3, n_a)

    # Cluster B: Pelanggan Potensial
    n_b = 80
    usia_b = rng.uniform(30, 50, n_b)
    pengeluaran_b = rng.uniform(500, 900, n_b)
    frekuensi_b = rng.uniform(4, 7, n_b)

    # Cluster C: Pelanggan Premium
    n_c = 60
    usia_c = rng.uniform(35, 60, n_c)
    pengeluaran_c = rng.uniform(1000, 2000, n_c)
    frekuensi_c = rng.uniform(8, 15, n_c)

    usia = np.concatenate([usia_a, usia_b, usia_c])
    pengeluaran = np.concatenate([pengeluaran_a, pengeluaran_b, pengeluaran_c])
    frekuensi = np.concatenate([frekuensi_a, frekuensi_b, frekuensi_c])

    # Tambah noise kecil agar tidak terlalu perfect
    usia += rng.normal(0, 1, len(usia))
    pengeluaran += rng.normal(0, 20, len(pengeluaran))
    frekuensi += rng.normal(0, 0.3, len(frekuensi))

    df = pd.DataFrame({
        "usia": np.round(usia, 1),
        "pengeluaran_bulanan": np.round(pengeluaran, 1),
        "frekuensi_transaksi": np.round(frekuensi, 1)
    })
    return df

# ── Fungsi: assign label interpretatif ──────────────────────
def assign_label_cluster(labels_array, df_asli, kolom_proxy, k, is_demo):
    """
    Ranking cluster berdasarkan rata-rata kolom_proxy.
    Tertinggi → Premium, Terendah → Pasif, Tengah → Potensial.
    """
    rata_per_cluster = {}
    for c in range(k):
        mask = labels_array == c
        if mask.sum() > 0:
            rata_per_cluster[c] = df_asli.loc[mask, kolom_proxy].mean()
        else:
            rata_per_cluster[c] = 0

    # Urutkan cluster dari pengeluaran tertinggi ke terendah
    sorted_clusters = sorted(rata_per_cluster.items(), key=lambda x: x[1], reverse=True)

    label_map = {}
    n_middle = k - 2  # jumlah cluster "Potensial"

    for rank, (cluster_id, _) in enumerate(sorted_clusters):
        if rank == 0:
            if is_demo:
                label_map[cluster_id] = "💎 Cluster Premium / Premium Customers"
            else:
                label_map[cluster_id] = "🔵 Cluster Tinggi / High"
        elif rank == len(sorted_clusters) - 1:
            if is_demo:
                label_map[cluster_id] = "😴 Cluster Pasif / Passive Customers"
            else:
                label_map[cluster_id] = "🔴 Cluster Rendah / Low"
        else:
            middle_rank = rank - 1  # index di antara Tinggi dan Rendah
            if n_middle == 1:
                if is_demo:
                    label_map[cluster_id] = "🌱 Cluster Potensial / Potential Customers"
                else:
                    label_map[cluster_id] = "🟡 Cluster Menengah / Medium"
            else:
                if is_demo:
                    label_map[cluster_id] = f"🌱 Cluster Potensial {middle_rank + 1}"
                else:
                    label_map[cluster_id] = f"🟡 Cluster Menengah {middle_rank + 1}"

    return label_map

# ── Fungsi: buat narasi cluster dinamis ─────────────────────
def buat_narasi_cluster(label, rata_rata_dict, is_demo):
    """Buat teks deskripsi dan rekomendasi berdasarkan label dan nilai rata-rata."""
    fitur_str = ", ".join([f"**{k}**: {v:.2f}" for k, v in rata_rata_dict.items()])

    if "Premium" in label or "Tinggi" in label:
        emoji = "💎"
        deskripsi = (
            f"Cluster ini memiliki profil tertinggi di antara semua kelompok. "
            f"Rata-rata fitur: {fitur_str}. "
            f"Pelanggan di cluster ini menunjukkan keterlibatan dan nilai transaksi yang sangat tinggi."
        )
        rekomendasi = (
            "💡 **Rekomendasi:** Pertahankan dengan program loyalitas eksklusif, "
            "penawaran VIP, dan layanan personal. Jangan kehilangan segmen ini."
        )
    elif "Pasif" in label or "Rendah" in label:
        emoji = "😴"
        deskripsi = (
            f"Cluster ini memiliki profil terendah di antara semua kelompok. "
            f"Rata-rata fitur: {fitur_str}. "
            f"Pelanggan di cluster ini jarang bertransaksi dan cenderung pasif."
        )
        rekomendasi = (
            "💡 **Rekomendasi:** Aktifkan kembali dengan diskon re-engagement, "
            "notifikasi promosi, atau kampanye win-back."
        )
    else:
        emoji = "🌱"
        deskripsi = (
            f"Cluster ini memiliki profil menengah. "
            f"Rata-rata fitur: {fitur_str}. "
            f"Pelanggan di cluster ini memiliki potensi untuk naik ke tier premium."
        )
        rekomendasi = (
            "💡 **Rekomendasi:** Dorong dengan program upsell dan reward bertahap. "
            "Target konversi ke tier premium melalui penawaran yang tepat sasaran."
        )

    return emoji, deskripsi, rekomendasi

# ── Sidebar ──────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Pengaturan")

    sumber_data = st.radio(
        "Pilih Sumber Data",
        options=["👥 Data Demo (Pelanggan Sintetis)", "📁 Upload CSV Sendiri"]
    )

    df_data = None
    fitur_cols = None
    is_demo = True

    # ── Opsi 1: Data Demo ────────────────────────────────────
    if sumber_data == "👥 Data Demo (Pelanggan Sintetis)":
        df_data = buat_data_demo()
        fitur_cols = ["usia", "pengeluaran_bulanan", "frekuensi_transaksi"]
        is_demo = True

    # ── Opsi 2: Upload CSV ───────────────────────────────────
    else:
        is_demo = False
        st.markdown(
            "**Format:** Minimal 2 kolom numerik, maksimal 10 kolom. "
            "Kolom non-numerik diabaikan otomatis."
        )
        file_upload = st.file_uploader("Upload file CSV", type=["csv"])

        if file_upload is not None:
            try:
                df_upload = pd.read_csv(file_upload)
                # Ambil hanya kolom numerik
                df_numerik = df_upload.select_dtypes(include=[np.number])
                if len(df_numerik.columns) < 2:
                    st.error("❌ File harus memiliki minimal 2 kolom numerik.")
                else:
                    kolom_numerik = list(df_numerik.columns[:10])
                    st.write("**Preview 5 baris pertama:**")
                    st.dataframe(df_upload.head())
                    fitur_terpilih = st.multiselect(
                        "Pilih kolom untuk clustering (min. 2)",
                        options=kolom_numerik,
                        default=kolom_numerik[:min(3, len(kolom_numerik))]
                    )
                    if len(fitur_terpilih) >= 2:
                        df_data = df_upload[fitur_terpilih].dropna().reset_index(drop=True)
                        fitur_cols = fitur_terpilih
                        st.success(f"✅ Data berhasil diproses: {len(df_data)} baris, {len(fitur_cols)} fitur")
                    else:
                        st.warning("⚠️ Pilih minimal 2 kolom untuk melanjutkan.")
            except Exception as e:
                st.error(f"❌ Gagal membaca file: {e}")

        # Fallback ke demo jika upload belum ada
        if df_data is None:
            df_data = buat_data_demo()
            fitur_cols = ["usia", "pengeluaran_bulanan", "frekuensi_transaksi"]
            is_demo = True

    st.divider()

    # Slider jumlah cluster
    k = st.slider("Jumlah Cluster (K)", min_value=2, max_value=6, value=3)

    # Checkbox interpretasi
    tampilkan_interpretasi = st.checkbox("💡 Tampilkan Interpretasi Cluster", value=False)

# ── Pastikan data tersedia ────────────────────────────────────
if df_data is None or fitur_cols is None:
    st.error("Data tidak tersedia.")
    st.stop()

# ── Proses: Scaling + KMeans ──────────────────────────────────
X = df_data[fitur_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_scaled)
df_data = df_data.copy()
df_data["cluster_id"] = labels

# ── Kolom proxy untuk interpretasi ───────────────────────────
if is_demo and "pengeluaran_bulanan" in fitur_cols:
    kolom_proxy = "pengeluaran_bulanan"
else:
    kolom_proxy = fitur_cols[0]

# ── Map label cluster ─────────────────────────────────────────
label_map = assign_label_cluster(labels, df_data, kolom_proxy, k, is_demo)

if tampilkan_interpretasi:
    df_data["cluster_label"] = df_data["cluster_id"].map(label_map)
else:
    df_data["cluster_label"] = df_data["cluster_id"].map(
        {c: f"Cluster {c}" for c in range(k)}
    )

# ── PCA 2D untuk visualisasi ──────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df_data["pca_x"] = X_pca[:, 0]
df_data["pca_y"] = X_pca[:, 1]

# Centroid PCA
centroid_scaled = kmeans.cluster_centers_
centroid_pca = pca.transform(centroid_scaled)

# ── Warna cluster (color-blind friendly) ─────────────────────
WARNA_CLUSTER = [
    "#2196F3",  # biru
    "#FF9800",  # oranye
    "#4CAF50",  # hijau
    "#E91E63",  # merah muda
    "#9C27B0",  # ungu
    "#00BCD4",  # cyan
]

# ── TABS ─────────────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs([
    "🗺️ Visualisasi Cluster",
    "📊 Profil Cluster",
    "📉 Elbow Method"
])

# ── Tab 1: Visualisasi Cluster ────────────────────────────────
with tab1:
    fig = go.Figure()

    for idx_c in range(k):
        mask = df_data["cluster_id"] == idx_c
        subset = df_data[mask]
        label_c = subset["cluster_label"].iloc[0] if len(subset) > 0 else f"Cluster {idx_c}"
        warna = WARNA_CLUSTER[idx_c % len(WARNA_CLUSTER)]

        # Teks hover: semua fitur asli
        hover_teks = [
            "<br>".join([f"{col}: {row[col]:.2f}" for col in fitur_cols])
            + f"<br><b>{label_c}</b>"
            for _, row in subset.iterrows()
        ]

        fig.add_trace(go.Scatter(
            x=subset["pca_x"],
            y=subset["pca_y"],
            mode="markers",
            name=label_c,
            marker=dict(color=warna, size=8, opacity=0.75),
            text=hover_teks,
            hovertemplate="%{text}<extra></extra>"
        ))

    # Centroid sebagai bintang
    for idx_c in range(k):
        label_c = label_map.get(idx_c, f"Cluster {idx_c}")
        fig.add_trace(go.Scatter(
            x=[centroid_pca[idx_c, 0]],
            y=[centroid_pca[idx_c, 1]],
            mode="markers",
            name=f"Centroid {idx_c}",
            marker=dict(
                symbol="star",
                size=20,
                color="black",
                line=dict(color="white", width=1.5)
            ),
            showlegend=False,
            hovertemplate=f"Centroid {label_c}<extra></extra>"
        ))

    fig.update_layout(
        title=f"Visualisasi Cluster (K={k}) — PCA 2D",
        xaxis_title="PCA Komponen 1",
        yaxis_title="PCA Komponen 2",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=500
    )
    st.plotly_chart(fig, use_container_width=True)
    st.caption(
        "📌 PCA digunakan hanya untuk visualisasi 2D. "
        "Proses clustering dilakukan pada data asli setelah scaling."
    )

# ── Tab 2: Profil Cluster ─────────────────────────────────────
with tab2:
    # Tabel rata-rata per cluster
    df_profil = df_data.groupby("cluster_label")[fitur_cols].mean().round(2)
    st.subheader("Rata-rata Fitur per Cluster")
    st.dataframe(df_profil, use_container_width=True)

    # Grouped bar chart
    df_profil_reset = df_profil.reset_index()
    fig_bar = go.Figure()

    for idx_c, row in df_profil_reset.iterrows():
        label_c = row["cluster_label"]
        warna = WARNA_CLUSTER[idx_c % len(WARNA_CLUSTER)]
        fig_bar.add_trace(go.Bar(
            name=label_c,
            x=fitur_cols,
            y=[row[col] for col in fitur_cols],
            marker_color=warna
        ))

    fig_bar.update_layout(
        title="Perbandingan Rata-rata Fitur per Cluster",
        barmode="group",
        xaxis_title="Fitur",
        yaxis_title="Nilai Rata-rata",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=400
    )
    st.plotly_chart(fig_bar, use_container_width=True)

    # Narasi interpretatif jika diaktifkan
    if tampilkan_interpretasi:
        st.subheader("📝 Interpretasi Cluster")
        for idx_c in range(k):
            label_c = label_map.get(idx_c, f"Cluster {idx_c}")
            mask = df_data["cluster_id"] == idx_c
            rata_dict = {
                col: df_data.loc[mask, col].mean()
                for col in fitur_cols
            }
            emoji, deskripsi, rekomendasi = buat_narasi_cluster(label_c, rata_dict, is_demo)

            with st.expander(f"{emoji} {label_c} ({mask.sum()} pelanggan)"):
                st.markdown(deskripsi)
                st.markdown(rekomendasi)

# ── Tab 3: Elbow Method ───────────────────────────────────────
with tab3:
    # Hitung inertia untuk K=1 sampai 10
    k_range = list(range(1, 11))
    inertia_list = []
    for k_test in k_range:
        km_test = KMeans(n_clusters=k_test, n_init=10, random_state=42)
        km_test.fit(X_scaled)
        inertia_list.append(km_test.inertia_)

    fig_elbow = go.Figure()

    # Garis biru: inertia
    fig_elbow.add_trace(go.Scatter(
        x=k_range,
        y=inertia_list,
        mode="lines+markers",
        name="Inertia",
        line=dict(color="royalblue", width=2),
        marker=dict(size=7, color="royalblue"),
        hovertemplate="K=%{x}<br>Inertia=%{y:.2f}<extra></extra>"
    ))

    # Marker merah: K yang sedang dipilih
    idx_k = k - 1  # index di k_range
    fig_elbow.add_trace(go.Scatter(
        x=[k],
        y=[inertia_list[idx_k]],
        mode="markers",
        name=f"K={k} (dipilih)",
        marker=dict(size=15, color="crimson", symbol="circle"),
        hovertemplate=f"K={k}<br>Inertia={inertia_list[idx_k]:.2f}<extra>K dipilih</extra>"
    ))

    fig_elbow.update_layout(
        title="Elbow Method — Mencari K Optimal",
        xaxis_title="Jumlah Cluster (K)",
        yaxis_title="Inertia",
        xaxis=dict(tickmode="linear", dtick=1),
        height=420
    )
    st.plotly_chart(fig_elbow, use_container_width=True)

    st.info(
        "📖 Cara membaca grafik ini: Pilih nilai K di titik 'siku' (elbow), "
        "yaitu titik di mana penurunan inertia mulai melambat secara signifikan. "
        "Menambah K setelah titik ini tidak memberikan manfaat yang sebanding."
    )

---
## 🧠 CELL 5 — Tulis File `app_cnn.py` (Deep Learning — Sesi 4)

In [ ]:
%%writefile app_cnn.py
# ============================================================
# APP 3: CNN IMAGE CLASSIFICATION DEMO — SESI 4
# Deep Learning dengan PyTorch
# ============================================================

import streamlit as st
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ── Konfigurasi halaman ──────────────────────────────────────
st.set_page_config(
    page_title="🧠 CNN Image Classification Demo",
    page_icon="🧠",
    layout="wide"
)

st.title("🧠 CNN Image Classification")
st.markdown(
    "Demo interaktif klasifikasi gambar menggunakan **Convolutional Neural Network (CNN)** "
    "dengan PyTorch. Latih model, coba prediksi gambar, dan bandingkan dengan ML klasik."
)

# ── Konstanta ────────────────────────────────────────────────
CIFAR10_CLASSES = [
    "✈️ Pesawat", "🚗 Mobil", "🐦 Burung", "🐱 Kucing", "🦌 Rusa",
    "🐶 Anjing", "🐸 Katak", "🐴 Kuda", "🚢 Kapal", "🚛 Truk"
]

# ── Definisi arsitektur CNN ──────────────────────────────────
class SimpleCNN(nn.Module):
    """CNN sederhana 2 blok konvolusi untuk CIFAR-10."""
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.blok1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1)
        )
        self.blok2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.blok1(x)
        x = self.blok2(x)
        x = self.classifier(x)
        return x


# ── Definisi MLP pembanding ──────────────────────────────────
class SimpleMLP(nn.Module):
    """MLP dasar tanpa konvolusi — sebagai pembanding CNN."""
    def __init__(self, num_classes=10):
        super(SimpleMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# ── Fungsi: load CIFAR-10 subset ────────────────────────────
@st.cache_data(show_spinner=False)
def load_cifar10_subset(n_train=1000, n_test=500):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010))
    ])
    train_full = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform
    )
    test_full = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=True, transform=transform
    )
    train_data   = torch.stack([train_full[i][0] for i in range(n_train)])
    train_labels = torch.tensor([train_full[i][1] for i in range(n_train)])
    test_data    = torch.stack([test_full[i][0] for i in range(n_test)])
    test_labels  = torch.tensor([test_full[i][1] for i in range(n_test)])
    return train_data, train_labels, test_data, test_labels


# ── Fungsi: training satu epoch ─────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_benar = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss   += loss.item() * len(y_batch)
        total_benar  += (output.argmax(1) == y_batch).sum().item()
    n = len(loader.dataset)
    return total_loss / n, total_benar / n


# ── Fungsi: evaluasi ─────────────────────────────────────────
def evaluasi(model, loader, criterion, device):
    model.eval()
    total_loss, total_benar = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            loss = criterion(output, y_batch)
            total_loss  += loss.item() * len(y_batch)
            total_benar += (output.argmax(1) == y_batch).sum().item()
    n = len(loader.dataset)
    return total_loss / n, total_benar / n


# ── Fungsi: prediksi satu gambar ────────────────────────────
def prediksi_gambar(model, img_tensor, device):
    model.eval()
    with torch.no_grad():
        output = model(img_tensor.unsqueeze(0).to(device))
        probs = torch.softmax(output, dim=1).squeeze().cpu().numpy()
    return probs


# ── Fungsi: denormalisasi untuk tampilan ────────────────────
def denormalisasi(tensor):
    mean = np.array([0.4914, 0.4822, 0.4465])
    std  = np.array([0.2023, 0.1994, 0.2010])
    img  = tensor.permute(1, 2, 0).numpy() * std + mean
    return np.clip(img, 0, 1)


# ── Sidebar ──────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Pengaturan Training")

    st.subheader("📦 Data")
    n_train = st.select_slider(
        "Jumlah Data Latih",
        options=[500, 1000, 2000, 5000], value=1000
    )
    n_test = st.select_slider(
        "Jumlah Data Uji",
        options=[200, 500, 1000], value=500
    )

    st.divider()
    st.subheader("🏗️ Arsitektur")
    pilih_model = st.selectbox(
        "Pilih Model",
        ["CNN (SimpleCNN)", "MLP (Tanpa Konvolusi)"],
        help="Bandingkan CNN vs MLP untuk melihat perbedaan kemampuan menangkap pola spasial"
    )

    st.divider()
    st.subheader("⚡ Hyperparameter")
    n_epoch        = st.slider("Jumlah Epoch", 1, 20, 5)
    batch_size     = st.selectbox("Batch Size", [32, 64, 128], index=1)
    learning_rate  = st.select_slider(
        "Learning Rate",
        options=[0.0001, 0.001, 0.01, 0.1], value=0.001
    )
    optimizer_nama = st.selectbox("Optimizer", ["Adam", "SGD"])

    st.divider()
    with st.expander("ℹ️ Panduan Hyperparameter"):
        st.markdown("""
**Epoch** — berapa kali model melihat seluruh data latih.
Lebih banyak epoch → potensi lebih akurat, tapi risiko overfit.

**Batch Size** — jumlah sampel per update gradien.
Kecil = update lebih sering (lambat tapi stabil).
Besar = update lebih jarang (cepat tapi bisa kurang presisi).

**Learning Rate** — seberapa besar langkah update bobot.
Terlalu besar → tidak konvergen. Terlalu kecil → lambat.

**Optimizer:**
- **Adam** — adaptif, cocok untuk pemula, lebih cepat konvergen.
- **SGD** — klasik, perlu tuning lebih, tapi bisa lebih general.
        """)

    tombol_latih = st.button("🚀 Mulai Training", type="primary", use_container_width=True)


# ── Device & load data ────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with st.spinner("⏳ Mengunduh dataset CIFAR-10 (hanya sekali)..."):
    train_data, train_labels, test_data, test_labels = load_cifar10_subset(n_train, n_test)

train_loader = DataLoader(TensorDataset(train_data, train_labels), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(test_data,  test_labels),  batch_size=batch_size, shuffle=False)

st.info(
    f"💻 Device: **{str(device).upper()}** | "
    f"Data latih: **{n_train}** gambar | "
    f"Data uji: **{n_test}** gambar | "
    f"Dataset: **CIFAR-10** (10 kelas objek)"
)

# ── Inisialisasi session state ────────────────────────────────
for key in ["model_cnn", "history", "model_nama"]:
    if key not in st.session_state:
        st.session_state[key] = None


# ── Proses training ───────────────────────────────────────────
if tombol_latih:
    model = SimpleCNN(10).to(device) if "CNN" in pilih_model else SimpleMLP(10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = (optim.Adam(model.parameters(), lr=learning_rate)
                 if optimizer_nama == "Adam"
                 else optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9))

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    progress_bar      = st.progress(0, text="Memulai training...")
    status_text       = st.empty()
    chart_placeholder = st.empty()

    for epoch in range(1, n_epoch + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, vl_acc = evaluasi(model, test_loader, criterion, device)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)

        progress_bar.progress(epoch / n_epoch, text=f"Epoch {epoch}/{n_epoch} — Val Acc: {vl_acc:.2%}")
        status_text.markdown(
            f"**Epoch {epoch}/{n_epoch}** | "
            f"Train Loss: `{tr_loss:.4f}` | Train Acc: `{tr_acc:.2%}` | "
            f"Val Loss: `{vl_loss:.4f}` | Val Acc: `{vl_acc:.2%}`"
        )
        ep_range = list(range(1, epoch + 1))
        fig_live = go.Figure()
        fig_live.add_trace(go.Scatter(x=ep_range, y=history["train_acc"],
            mode="lines+markers", name="Train Acc", line=dict(color="royalblue")))
        fig_live.add_trace(go.Scatter(x=ep_range, y=history["val_acc"],
            mode="lines+markers", name="Val Acc", line=dict(color="seagreen", dash="dash")))
        fig_live.update_layout(title="Akurasi Training (Live)", xaxis_title="Epoch",
            yaxis_title="Accuracy", yaxis=dict(range=[0, 1]), height=300,
            legend=dict(orientation="h", y=1.1))
        chart_placeholder.plotly_chart(fig_live, use_container_width=True)

    st.session_state["model_cnn"]  = model
    st.session_state["history"]    = history
    st.session_state["model_nama"] = pilih_model
    st.success(f"✅ Training selesai! Val Accuracy akhir: **{history['val_acc'][-1]:.2%}**")


# ── TABS ─────────────────────────────────────────────────────
tab1, tab2, tab3, tab4 = st.tabs([
    "📈 Hasil Training",
    "🔍 Prediksi Gambar",
    "🏗️ Arsitektur CNN",
    "⚖️ CNN vs ML Klasik"
])


# ────────────────────────────────────────────────────────────
# Tab 1: Hasil Training
# ────────────────────────────────────────────────────────────
with tab1:
    if st.session_state["history"] is None:
        st.info("⬅️ Atur parameter di sidebar lalu klik **Mulai Training** untuk melihat hasil.")
    else:
        history      = st.session_state["history"]
        ep_range     = list(range(1, len(history["train_acc"]) + 1))

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Train Accuracy", f"{history['train_acc'][-1]:.2%}")
        c2.metric("Val Accuracy",   f"{history['val_acc'][-1]:.2%}")
        c3.metric("Train Loss",     f"{history['train_loss'][-1]:.4f}")
        c4.metric("Val Loss",       f"{history['val_loss'][-1]:.4f}")

        fig_acc = go.Figure()
        fig_acc.add_trace(go.Scatter(x=ep_range, y=history["train_acc"],
            mode="lines+markers", name="Train Accuracy", line=dict(color="royalblue", width=2)))
        fig_acc.add_trace(go.Scatter(x=ep_range, y=history["val_acc"],
            mode="lines+markers", name="Val Accuracy", line=dict(color="seagreen", width=2, dash="dash")))
        fig_acc.update_layout(
            title=f"Akurasi per Epoch — {st.session_state['model_nama']}",
            xaxis_title="Epoch", yaxis_title="Accuracy",
            xaxis=dict(tickmode="linear", dtick=1),
            yaxis=dict(range=[0, 1]),
            legend=dict(orientation="h", y=1.1), height=380
        )
        st.plotly_chart(fig_acc, use_container_width=True)

        fig_loss = go.Figure()
        fig_loss.add_trace(go.Scatter(x=ep_range, y=history["train_loss"],
            mode="lines+markers", name="Train Loss", line=dict(color="crimson", width=2)))
        fig_loss.add_trace(go.Scatter(x=ep_range, y=history["val_loss"],
            mode="lines+markers", name="Val Loss", line=dict(color="darkorange", width=2, dash="dash")))
        fig_loss.update_layout(
            title="Loss per Epoch", xaxis_title="Epoch", yaxis_title="Loss",
            xaxis=dict(tickmode="linear", dtick=1),
            legend=dict(orientation="h", y=1.1), height=350
        )
        st.plotly_chart(fig_loss, use_container_width=True)

        gap = history["train_acc"][-1] - history["val_acc"][-1]
        if gap > 0.15:
            st.warning(
                f"⚠️ **Potensi Overfitting** — Gap train vs val: `{gap:.2%}`. "
                "Coba kurangi epoch, tambah dropout, atau perbanyak data."
            )
        elif history["val_acc"][-1] < 0.3:
            st.warning(
                "⚠️ **Akurasi rendah (Underfitting)** — Coba tambah epoch, "
                "naikkan learning rate, atau gunakan lebih banyak data."
            )
        else:
            st.success("✅ Training berjalan normal — tidak ada tanda overfitting yang signifikan.")


# ────────────────────────────────────────────────────────────
# Tab 2: Prediksi Gambar
# ────────────────────────────────────────────────────────────
with tab2:
    if st.session_state["model_cnn"] is None:
        st.info("⬅️ Latih model terlebih dahulu sebelum mencoba prediksi.")
    else:
        model = st.session_state["model_cnn"]

        st.subheader("Pilih Sumber Gambar")

        # ── Dua opsi input yang jelas ────────────────────────
        opsi = st.radio(
            "",
            ["🎲 Gunakan Sampel dari Dataset CIFAR-10",
             "📁 Upload Gambar Sendiri"],
            horizontal=True,
            label_visibility="collapsed"
        )

        img_tensor = None
        label_asli = None

        # ── Opsi 1: Sampel CIFAR-10 ──────────────────────────
        if opsi == "🎲 Gunakan Sampel dari Dataset CIFAR-10":
            st.markdown(
                "Dataset CIFAR-10 berisi 10 kelas: "
                "✈️ Pesawat, 🚗 Mobil, 🐦 Burung, 🐱 Kucing, 🦌 Rusa, "
                "🐶 Anjing, 🐸 Katak, 🐴 Kuda, 🚢 Kapal, 🚛 Truk"
            )

            col_filter, col_acak = st.columns([2, 1])
            with col_filter:
                filter_kelas = st.selectbox(
                    "Filter berdasarkan kelas (opsional)",
                    ["Semua Kelas"] + CIFAR10_CLASSES
                )
            with col_acak:
                st.markdown("<br>", unsafe_allow_html=True)
                acak = st.button("🎲 Acak Gambar", use_container_width=True)

            # Tentukan indeks valid berdasarkan filter kelas
            if filter_kelas == "Semua Kelas":
                indeks_valid = list(range(len(test_data)))
            else:
                kelas_idx = CIFAR10_CLASSES.index(filter_kelas)
                indeks_valid = [i for i in range(len(test_labels))
                                if test_labels[i].item() == kelas_idx]

            # Simpan indeks terpilih di session state
            if acak or "idx_sampel" not in st.session_state:
                if indeks_valid:
                    st.session_state["idx_sampel"] = int(
                        np.random.choice(indeks_valid)
                    )

            # Pastikan indeks masih valid setelah filter berubah
            if st.session_state.get("idx_sampel") not in indeks_valid:
                st.session_state["idx_sampel"] = indeks_valid[0] if indeks_valid else 0

            idx_terpilih = st.session_state["idx_sampel"]

            # Slider untuk navigasi manual
            idx_slider = st.slider(
                "Atau pilih indeks gambar secara manual",
                min_value=0,
                max_value=len(indeks_valid) - 1,
                value=indeks_valid.index(idx_terpilih) if idx_terpilih in indeks_valid else 0,
                help="Geser untuk melihat gambar lain"
            )
            idx_terpilih = indeks_valid[idx_slider]
            st.session_state["idx_sampel"] = idx_terpilih

            img_tensor  = test_data[idx_terpilih]
            label_asli  = test_labels[idx_terpilih].item()

            # Tampilkan gambar langsung (diperbesar)
            img_display = denormalisasi(img_tensor)
            col_img, col_meta = st.columns([1, 3])
            with col_img:
                # Perbesar gambar 32x32 → tampil 160px
                pil_besar = Image.fromarray((img_display * 255).astype(np.uint8)).resize(
                    (160, 160), Image.NEAREST
                )
                st.image(pil_besar, caption="Gambar asli (32×32, diperbesar)")
            with col_meta:
                st.markdown(f"**Indeks:** `{idx_terpilih}`")
                st.markdown(f"**Kelas asli:** {CIFAR10_CLASSES[label_asli]}")
                st.markdown(f"**Ukuran asli:** 32 × 32 piksel × 3 channel (RGB)")

        # ── Opsi 2: Upload Gambar Sendiri ────────────────────
        else:
            st.markdown(
                "Upload gambar dari perangkat Anda. "
                "Model akan otomatis meresize ke **32×32 piksel** sesuai format CIFAR-10."
            )
            st.caption(
                "💡 Tips: Untuk hasil terbaik, gunakan gambar yang termasuk salah satu "
                "dari 10 kelas CIFAR-10 dengan objek yang jelas dan latar tidak terlalu ramai."
            )

            file_up = st.file_uploader(
                "Pilih file gambar (JPG / PNG)",
                type=["jpg", "jpeg", "png"],
                help="Gambar akan otomatis diresize ke 32×32 piksel"
            )

            if file_up is not None:
                pil_ori  = Image.open(file_up).convert("RGB")
                pil_kecil = pil_ori.resize((32, 32))
                # Tampilkan dua versi: asli dan 32x32
                col_ori, col_kecil, col_info = st.columns([2, 1, 2])
                with col_ori:
                    st.image(pil_ori, caption=f"Gambar asli ({pil_ori.width}×{pil_ori.height}px)", use_container_width=True)
                with col_kecil:
                    pil_besar = pil_kecil.resize((160, 160), Image.NEAREST)
                    st.image(pil_besar, caption="Setelah resize (32×32)")
                with col_info:
                    st.markdown("**Proses yang terjadi:**")
                    st.markdown(f"1. Gambar asli: `{pil_ori.width}×{pil_ori.height}` px")
                    st.markdown("2. Resize → `32×32` px")
                    st.markdown("3. Normalisasi nilai piksel")
                    st.markdown("4. Masuk ke CNN → probabilitas 10 kelas")

                transform_infer = transforms.Compose([
                    transforms.ToTensor(),
                    transforms.Normalize((0.4914, 0.4822, 0.4465),
                                         (0.2023, 0.1994, 0.2010))
                ])
                img_tensor = transform_infer(pil_kecil)
                st.warning(
                    "⚠️ Model dilatih khusus pada dataset CIFAR-10. "
                    "Akurasi pada gambar di luar distribusi dataset bisa lebih rendah — "
                    "ini adalah fenomena normal yang disebut **distribusi shift**."
                )
            else:
                st.info("⬆️ Belum ada gambar yang diupload.")

        # ── Hasil prediksi (berlaku untuk kedua opsi) ────────
        if img_tensor is not None:
            st.divider()
            st.subheader("🎯 Hasil Prediksi")

            probs    = prediksi_gambar(model, img_tensor, device)
            pred_idx = int(np.argmax(probs))

            col_hasil, col_chart = st.columns([1, 2])
            with col_hasil:
                st.metric("Prediksi Model", CIFAR10_CLASSES[pred_idx])
                st.metric("Confidence", f"{probs[pred_idx]:.2%}")
                if label_asli is not None:
                    if pred_idx == label_asli:
                        st.success("✅ Prediksi **BENAR**")
                    else:
                        st.error(f"❌ Prediksi **SALAH**\n\nKelas asli: {CIFAR10_CLASSES[label_asli]}")

            with col_chart:
                warna_bar = [
                    "#4CAF50" if i == pred_idx else
                    ("#2196F3" if label_asli is not None and i == label_asli else "#E0E0E0")
                    for i in range(10)
                ]
                fig_prob = go.Figure(go.Bar(
                    x=[CIFAR10_CLASSES[i] for i in range(10)],
                    y=probs,
                    marker_color=warna_bar,
                    text=[f"{p:.1%}" for p in probs],
                    textposition="outside",
                    hovertemplate="%{x}<br>Probabilitas: %{y:.2%}<extra></extra>"
                ))
                fig_prob.update_layout(
                    title="Distribusi Probabilitas per Kelas",
                    xaxis_title="Kelas", yaxis_title="Probabilitas",
                    yaxis=dict(range=[0, 1.1]),
                    height=400
                )
                st.plotly_chart(fig_prob, use_container_width=True)
                if label_asli is not None:
                    st.caption("🟢 Hijau = prediksi model | 🔵 Biru = kelas asli")

        # ── Galeri contoh per kelas ──────────────────────────
        with st.expander("🖼️ Galeri Contoh Gambar CIFAR-10 (1 sampel per kelas)"):
            cols = st.columns(10)
            for cls_idx in range(10):
                for i in range(len(test_labels)):
                    if test_labels[i].item() == cls_idx:
                        img_ex   = denormalisasi(test_data[i])
                        pil_ex   = Image.fromarray((img_ex * 255).astype(np.uint8)).resize(
                            (64, 64), Image.NEAREST
                        )
                        cols[cls_idx].image(pil_ex, caption=CIFAR10_CLASSES[cls_idx],
                                            use_container_width=True)
                        break


# ────────────────────────────────────────────────────────────
# Tab 3: Arsitektur CNN
# ────────────────────────────────────────────────────────────
with tab3:
    st.subheader("🏗️ Diagram Arsitektur SimpleCNN")

    arsitektur = [
        {"Layer": "Input",              "Tipe": "—",                      "Detail": "3 × 32 × 32 (RGB)",                    "Output Shape": "3 × 32 × 32"},
        {"Layer": "Conv2d-1",           "Tipe": "Konvolusi",               "Detail": "32 filter, kernel 3×3, padding 1",     "Output Shape": "32 × 32 × 32"},
        {"Layer": "BatchNorm + ReLU",   "Tipe": "Normalisasi + Aktivasi",  "Detail": "Normalisasi per batch, non-linear",    "Output Shape": "32 × 32 × 32"},
        {"Layer": "Conv2d-2",           "Tipe": "Konvolusi",               "Detail": "32 filter, kernel 3×3, padding 1",     "Output Shape": "32 × 32 × 32"},
        {"Layer": "MaxPool2d",          "Tipe": "Pooling",                 "Detail": "Kernel 2×2, stride 2",                 "Output Shape": "32 × 16 × 16"},
        {"Layer": "Dropout2d (p=0.1)",  "Tipe": "Regularisasi",            "Detail": "10% channel di-nol-kan secara acak",   "Output Shape": "32 × 16 × 16"},
        {"Layer": "Conv2d-3",           "Tipe": "Konvolusi",               "Detail": "64 filter, kernel 3×3, padding 1",     "Output Shape": "64 × 16 × 16"},
        {"Layer": "BatchNorm + ReLU",   "Tipe": "Normalisasi + Aktivasi",  "Detail": "Normalisasi per batch, non-linear",    "Output Shape": "64 × 16 × 16"},
        {"Layer": "Conv2d-4",           "Tipe": "Konvolusi",               "Detail": "64 filter, kernel 3×3, padding 1",     "Output Shape": "64 × 16 × 16"},
        {"Layer": "MaxPool2d",          "Tipe": "Pooling",                 "Detail": "Kernel 2×2, stride 2",                 "Output Shape": "64 × 8 × 8"},
        {"Layer": "Flatten",            "Tipe": "Reshape",                 "Detail": "64 × 8 × 8 → vektor 1D",              "Output Shape": "4.096"},
        {"Layer": "Linear-1",           "Tipe": "Fully Connected",         "Detail": "4.096 → 256 neuron",                  "Output Shape": "256"},
        {"Layer": "Dropout (p=0.4)",    "Tipe": "Regularisasi",            "Detail": "40% neuron di-nol-kan secara acak",    "Output Shape": "256"},
        {"Layer": "Linear-2 (Output)",  "Tipe": "Fully Connected",         "Detail": "256 → 10 neuron (1 per kelas)",        "Output Shape": "10"},
        {"Layer": "Softmax",            "Tipe": "Aktivasi Output",         "Detail": "Konversi logit → probabilitas",        "Output Shape": "10 probabilitas"},
    ]
    st.dataframe(pd.DataFrame(arsitektur), use_container_width=True, hide_index=True)

    st.subheader("📊 Alur Dimensi Data per Layer")
    layer_names = ["Input\n3×32×32", "Blok1 Conv\n32×32×32", "Blok1 Pool\n32×16×16",
                   "Blok2 Conv\n64×16×16", "Blok2 Pool\n64×8×8",
                   "Flatten\n4096", "FC-1\n256", "Output\n10"]
    layer_sizes = [3*32*32, 32*32*32, 32*16*16, 64*16*16, 64*8*8, 4096, 256, 10]
    warna_layer = ["#2196F3","#4CAF50","#4CAF50","#FF9800","#FF9800","#9C27B0","#E91E63","#F44336"]

    fig_dim = go.Figure(go.Bar(
        x=layer_names, y=layer_sizes, marker_color=warna_layer,
        hovertemplate="%{x}<br>Jumlah nilai: %{y:,}<extra></extra>"
    ))
    fig_dim.update_layout(
        title="Jumlah Nilai per Layer (skala log)",
        xaxis_title="Layer", yaxis_title="Jumlah Nilai",
        yaxis_type="log", height=380
    )
    st.plotly_chart(fig_dim, use_container_width=True)

    st.subheader("📚 Konsep Kunci CNN")
    ca, cb, cc = st.columns(3)
    with ca:
        st.markdown("""
**🔲 Konvolusi (Conv2d)**
Filter kecil bergeser di atas gambar untuk mendeteksi pola lokal.
- Layer awal → tepi, warna
- Layer dalam → bentuk, tekstur kompleks
- **Parameter sharing**: filter yang sama dipakai di semua posisi → jauh lebih efisien dari MLP
        """)
    with cb:
        st.markdown("""
**📉 Max Pooling**
Ambil nilai maksimum dari tiap region kecil → reduksi dimensi.
- Kurangi parameter secara drastis
- Buat representasi lebih ringkas
- **Translasi invariance**: objek yang sedikit bergeser tetap terdeteksi
        """)
    with cc:
        st.markdown("""
**🛡️ Batch Norm + Dropout**
- **BatchNorm**: normalisasi aktivasi per batch → training lebih stabil
- **Dropout**: matikan neuron secara acak → cegah overfit, paksa model belajar lebih robust
        """)

    if st.session_state["model_cnn"] is not None:
        m = st.session_state["model_cnn"]
        total = sum(p.numel() for p in m.parameters())
        train = sum(p.numel() for p in m.parameters() if p.requires_grad)
        st.info(f"🔢 **Parameter {st.session_state['model_nama']}:** Total = `{total:,}` | Trainable = `{train:,}`")
    else:
        total = sum(p.numel() for p in SimpleCNN().parameters())
        st.info(f"🔢 **Parameter SimpleCNN (default):** Total = `{total:,}`")


# ────────────────────────────────────────────────────────────
# Tab 4: CNN vs ML Klasik
# ────────────────────────────────────────────────────────────
with tab4:
    st.subheader("⚖️ Perbandingan CNN vs ML Klasik pada CIFAR-10")
    st.markdown(
        "Benchmark ini melatih **SVM** dan **Random Forest** pada fitur piksel mentah "
        "(gambar di-*flatten* jadi vektor 1D), lalu dibandingkan dengan CNN yang sudah Anda latih."
    )

    n_compare    = st.slider("Jumlah data untuk benchmark ML Klasik", 200, 1000, 500,
                             help="ML klasik lebih lambat — gunakan data lebih sedikit untuk perbandingan cepat")
    tombol_bench = st.button("⚡ Jalankan Benchmark", type="secondary")

    if tombol_bench:
        X_tr = train_data[:n_compare].reshape(n_compare, -1).numpy()
        y_tr = train_labels[:n_compare].numpy()
        X_te = test_data[:n_compare].reshape(n_compare, -1).numpy()
        y_te = test_labels[:n_compare].numpy()

        hasil = {}
        with st.spinner("Training SVM (RBF)..."):
            svm = SVC(kernel="rbf", C=1.0, random_state=42)
            svm.fit(X_tr, y_tr)
            hasil["SVM (RBF)"] = accuracy_score(y_te, svm.predict(X_te))

        with st.spinner("Training Random Forest..."):
            rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
            rf.fit(X_tr, y_tr)
            hasil["Random Forest"] = accuracy_score(y_te, rf.predict(X_te))

        if st.session_state["model_cnn"] is not None:
            crit  = nn.CrossEntropyLoss()
            ldr   = DataLoader(TensorDataset(test_data[:n_compare], test_labels[:n_compare]), batch_size=64)
            _, acc = evaluasi(st.session_state["model_cnn"], ldr, crit, device)
            hasil[st.session_state["model_nama"]] = acc

        st.session_state["benchmark"] = hasil

    if st.session_state.get("benchmark"):
        hasil       = st.session_state["benchmark"]
        nama_list   = list(hasil.keys())
        acc_list    = list(hasil.values())

        warna = ["#4CAF50" if "CNN" in n else ("#FF9800" if "MLP" in n else "#2196F3")
                 for n in nama_list]

        fig_b = go.Figure(go.Bar(
            x=nama_list, y=acc_list, marker_color=warna,
            text=[f"{v:.2%}" for v in acc_list], textposition="outside",
            hovertemplate="%{x}<br>Akurasi: %{y:.2%}<extra></extra>"
        ))
        fig_b.update_layout(
            title="Perbandingan Akurasi: CNN vs ML Klasik (Test Set)",
            xaxis_title="Model", yaxis_title="Akurasi",
            yaxis=dict(range=[0, 1.1]), height=420
        )
        st.plotly_chart(fig_b, use_container_width=True)

        df_b = pd.DataFrame({
            "Model":  nama_list,
            "Akurasi": [f"{v:.2%}" for v in acc_list],
            "Tipe":   ["Deep Learning" if ("CNN" in n or "MLP" in n) else "ML Klasik"
                       for n in nama_list]
        })
        st.dataframe(df_b, use_container_width=True, hide_index=True)

        st.subheader("💡 Insight")
        best     = max(hasil, key=hasil.get)
        ml_vals  = {k: v for k, v in hasil.items() if "CNN" not in k and "MLP" not in k}
        if ml_vals:
            best_ml, best_ml_acc = max(ml_vals.items(), key=lambda x: x[1])
            if "CNN" in best:
                selisih = hasil[best] - best_ml_acc
                st.success(
                    f"🏆 **CNN unggul {selisih:.2%}** dibanding ML klasik terbaik "
                    f"({best_ml}: {best_ml_acc:.2%}). "
                    "CNN mampu menangkap pola spasial gambar yang tidak bisa dilakukan "
                    "SVM/RF yang hanya melihat piksel mentah."
                )
            else:
                st.info(
                    f"📊 Model terbaik saat ini: **{best}** ({hasil[best]:.2%}). "
                    "CNN mungkin perlu lebih banyak epoch atau data. Coba latih lebih lama!"
                )

        st.markdown("""
**Mengapa CNN lebih unggul untuk data gambar?**

| Aspek | ML Klasik (SVM / RF) | CNN |
|-------|----------------------|-----|
| Format input | Vektor piksel datar (flat) | Tensor spasial H × W × C |
| Cara ekstraksi fitur | Manual / piksel mentah | Otomatis dipelajari (feature learning) |
| Pola spasial | ❌ Tidak bisa menangkap | ✅ Konvolusi menangkap pola lokal |
| Translasi invariance | ❌ Tidak ada | ✅ Max Pooling memberi invariance |
| Skalabilitas | ❌ Lambat di gambar besar | ✅ Parameter sharing sangat efisien |
        """)

---
## 🚀 CELL 6 — Jalankan Semua App (3 Aplikasi Sekaligus)

> Jalankan cell ini untuk menghidupkan ketiga app secara bersamaan.
> Tunggu hingga ketiga URL muncul, lalu klik masing-masing untuk membuka di tab baru.

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok

APPS = [
    ("app_timeseries.py", 8501, "🔮 TIME SERIES FORECASTING"),
    ("app_segmentasi.py", 8502, "👥 CUSTOMER SEGMENTATION"),
    ("app_cnn.py",        8503, "🧠 CNN IMAGE CLASSIFICATION"),
]

def jalankan(nama_file, port):
    subprocess.Popen(
        ["streamlit", "run", nama_file,
         "--server.port", str(port),
         "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

# Jalankan ketiga app di background thread
for nama_file, port, _ in APPS:
    threading.Thread(target=jalankan, args=(nama_file, port), daemon=True).start()

print("⏳ Menunggu Streamlit siap...")
time.sleep(8)

# Buat tunnel ngrok untuk masing-masing port
tunnels = {}
for nama_file, port, label in APPS:
    t = ngrok.connect(port)
    tunnels[label] = t.public_url

print()
print("=" * 60)
print("  ✅ SEMUA APP SIAP DIGUNAKAN")
print("=" * 60)
for label, url in tunnels.items():
    print(f"  {label}")
    print(f"  🌐 {url}")
    print()
print("=" * 60)
print("  Klik URL di atas untuk membuka app di tab browser baru.")
print("=" * 60)


---
## 🛑 CELL 7 — Hentikan Semua Tunnel Ngrok

> **Jangan jalankan cell ini saat Run All!**
>
> Cell ini sengaja dikomentari agar tidak ikut berjalan saat **Runtime → Run all**.
> Jalankan **hanya** jika ingin menghentikan semua tunnel ngrok secara manual.
>
> **Cara pakai:** Hapus tanda `#` pada baris kode di bawah, lalu klik ▶ jalankan cell ini.

In [ ]:
# Hapus tanda # di bawah ini, lalu jalankan cell ini untuk menghentikan semua tunnel

# ngrok.kill()
# print("✅ Semua tunnel ngrok telah dihentikan.")